# Manufacturing KNN – Development Pipeline Notebook

This notebook defines Kubeflow Pipeline components, compiles the pipeline, and submits a DEV run on Vertex AI.

In [14]:

from kfp.dsl import component, Dataset, Input, Output, Model, Metrics, Artifact
from kfp import dsl, compiler
from google.cloud import aiplatform
import time

project = !gcloud config get-value project
PROJECT_ID = project[0]
PROJECT_ID = "mlops-pipeline-01"
REGION = "europe-west3"
BUCKET_NAME = PROJECT_ID


MODEL_DISPLAY_NAME = "mlops-model-dev"
ENDPOINT_NAME = "mlops-endpoint-dev"


print("Project:", PROJECT_ID)
print("Bucket:", BUCKET_NAME)
print("Region:", REGION)

aiplatform.init(project=PROJECT_ID, location=REGION)


Project: mlops-pipeline-01
Bucket: mlops-pipeline-01
Region: europe-west3


In [15]:
@component(
    base_image="python:3.10",
    packages_to_install=["pandas", "google-cloud-storage<3"],
)
def extract_data_op(bucket_name: str, raw_data: Output[Dataset]):
    import pandas as pd
    from google.cloud import storage
    import io, os

    client = storage.Client()
    bucket = client.bucket(BUCKET_NAME)
    dfs = []

    for blob in bucket.list_blobs():
        if blob.name.endswith("_breakdowns.csv"):
            df = pd.read_csv(io.BytesIO(blob.download_as_string()))
            if "BREAKS" in df.columns:
                dfs.append(df)

    if not dfs:
        raise RuntimeError("No valid CSV files found")

    merged = pd.concat(dfs, ignore_index=True)
    os.makedirs(raw_data.path, exist_ok=True)
    merged.to_csv(f"{raw_data.path}/raw_data.csv", index=False)
    print(f"Raw data written to {output_path}")


In [16]:
@component(
    base_image="python:3.10",
    packages_to_install=["pandas"],
)
def prepare_data_op(raw_data: Input[Dataset], prepared_data: Output[Dataset]):
    import pandas as pd, os
    df = pd.read_csv(f"{raw_data.path}/raw_data.csv")
    rows = []
    for _, r in df.iterrows():
        rows.append({
            "job_id": r.iloc[0],
            "priority": r.iloc[1],
            "processing_time": r.iloc[19],
            "breaks": r.iloc[22],
        })

    out = pd.DataFrame(rows)
    os.makedirs(prepared_data.path, exist_ok=True)
    out.to_csv(f"{prepared_data.path}/prepared_data.csv", index=False)


In [17]:
@component(
    base_image="python:3.10",
    packages_to_install=["pandas", "scikit-learn", "joblib"],
)
def train_knn_op(prepared_data: Input[Dataset], model: Output[Model]):
    import pandas as pd, joblib, os
    from sklearn.neighbors import KNeighborsClassifier

    df = pd.read_csv(f"{prepared_data.path}/prepared_data.csv")
    X, y = df.drop("breaks", axis=1), df["breaks"]

    clf = KNeighborsClassifier(n_neighbors=5,metric="minkowski")
    clf.fit(X, y)

    os.makedirs(model.path, exist_ok=True)
    joblib.dump(clf, f"{model.path}/model.joblib")
    print("Model training completed")
    print(f"Model saved to: {model_path}")

In [18]:

from typing import NamedTuple

@component(
    base_image="python:3.10",
    packages_to_install=["pandas", "scikit-learn", "joblib"],
)
def evaluate_model_op(
    model: Input[Model],
    prepared_data: Input[Dataset],
    metrics: Output[Metrics],
    f1_threshold: float = 0.85,
) -> NamedTuple("Outputs", [("deploy_decision", str)]):

    import pandas as pd, joblib
    from sklearn.metrics import f1_score

    df = pd.read_csv(f"{prepared_data.path}/prepared_data.csv")
    X, y = df.drop("breaks", axis=1), df["breaks"]

    clf = joblib.load(f"{model.path}/model.joblib")
    preds = clf.predict(X)

    f1 = f1_score(y, preds, average="weighted")
    metrics.log_metric("f1_weighted", f1)

    return ("true" if f1 >= f1_threshold else "false",)


In [19]:

@component(
    base_image="python:3.10",
    packages_to_install=["google-cloud-aiplatform"],
)
def register_model_op(
    project_id: str,
    location: str,
    model: Input[Model],
    display_name: str,
):
    from google.cloud import aiplatform
    aiplatform.init(project=project_id, location=location)

    aiplatform.Model.upload(
        display_name=display_name,
        artifact_uri=model.path,
        serving_container_image_uri="europe-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-0:latest",
        sync=True,
    )


In [20]:
@dsl.pipeline(name="mlops-manufacturing-dev")
def mlops_manufacturing_pipeline(
    project_id: str,
    location: str,
    bucket_name: str,
    model_display_name: str,
    f1_threshold: float = 0.85,
):

    extract = extract_data_op(bucket_name=BUCKET_NAME)
    prepare = prepare_data_op(raw_data=extract.outputs["raw_data"])
    train = train_knn_op(prepared_data=prepare.outputs["prepared_data"])
    eval = evaluate_model_op(
        model=train.outputs["model"],
        prepared_data=prepare.outputs["prepared_data"],
        f1_threshold=f1_threshold,
    )

    with dsl.If(eval.outputs["deploy_decision"] == "true"):
        register_model_op(
            project_id=project_id,
            location=location,
            model=train.outputs["model"],
            display_name=model_display_name,
        )


In [22]:

compiler.Compiler().compile(
    pipeline_func=mlops_manufacturing_pipeline,
    package_path="mlops_manufacturing_pipeline.yaml",
)


In [23]:

from google.cloud.aiplatform.pipeline_jobs import PipelineJob

PIPELINE_ROOT = f"gs://{BUCKET_NAME}/pipeline-root-dev-{int(time.time())}"

job = PipelineJob(
    display_name="mlops-manufacturing-dev",
    template_path="mlops_manufacturing_pipeline.yaml",
    pipeline_root=PIPELINE_ROOT,
    parameter_values={
        "project_id": PROJECT_ID,
        "location": REGION,
        "bucket_name": BUCKET_NAME,
        "model_display_name": MODEL_DISPLAY_NAME,
        "f1_threshold": 0.85,
    },
)

job.run(sync=True)


Creating PipelineJob
PipelineJob created. Resource name: projects/71707089683/locations/europe-west3/pipelineJobs/mlops-manufacturing-dev-20260119210042
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/71707089683/locations/europe-west3/pipelineJobs/mlops-manufacturing-dev-20260119210042')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/europe-west3/pipelines/runs/mlops-manufacturing-dev-20260119210042?project=71707089683
PipelineJob projects/71707089683/locations/europe-west3/pipelineJobs/mlops-manufacturing-dev-20260119210042 current state:
PipelineState.PIPELINE_STATE_RUNNING
PipelineJob projects/71707089683/locations/europe-west3/pipelineJobs/mlops-manufacturing-dev-20260119210042 current state:
PipelineState.PIPELINE_STATE_RUNNING
PipelineJob projects/71707089683/locations/europe-west3/pipelineJobs/mlops-manufacturing-dev-20260119210042 current state:
PipelineState.PIPELINE_STATE_RUNNING
PipelineJob projects/71

RuntimeError: Job failed with:
code: 9
message: " The DAG failed because some tasks failed. The failed tasks are: [extract-data-op].; Job (project_id = mlops-pipeline-01, job_id = 6928534039523491840) is failed due to the above error.; Failed to handle the job: {project_number = 71707089683, job_id = 6928534039523491840}"
